In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# load dataset
df = pd.read_csv("customer_reviews.csv")
df

,review_id,product_name,category,rating,review_text,true_sentiment,reviewer_age,helpful_votes
0,1,Laptop,Books,5.0,Super happy with this purchase. Packaging was ...,Positive,35,45
1,2,Laptop,Home & Kitchen,5.0,Works like a charm. Very happy with the overal...,Positive,61,16
2,3,Jeans,Beauty,5.0,Fantastic product at a very reasonable price p...,Positive,27,41
3,4,Blender,Sports,5.0,Works like a charm. Very happy with the overal...,Positive,22,11
4,5,Yoga Mat,Books,4.0,Works like a charm. Very happy with the overal...,Positive,26,47
...,...,...,...,...,...,...,...,...
1225,832,Novel,Electronics,1.0,Horrible experience from order to delivery. Av...,Negative,29,10
1226,582,Python Guide,Electronics,5.0,Great value for money. The product works perfe...,Positive,47,35
1227,199,Jeans,Books,4.0,Excellent build quality and quick shipping. Fi...,Positive,47,36
1228,917,Python Guide,Beauty,1.0,Very disappointed. Product broke after just tw...,Negative,51,45


In [4]:
# Dataset Shape
print("Rows :", df.shape[0])
print("Columns :", df.shape[1])

Rows : 1230
Columns : 8


In [5]:
# Dataset Information
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1230 entries, 0 to 1229
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   review_id       1230 non-null   int64  
 1   product_name    1230 non-null   object 
 2   category        1230 non-null   object 
 3   rating          1220 non-null   float64
 4   review_text     1210 non-null   object 
 5   true_sentiment  1230 non-null   object 
 6   reviewer_age    1230 non-null   int64  
 7   helpful_votes   1230 non-null   int64  
dtypes: float64(1), int64(3), object(4)
memory usage: 77.0+ KB


In [6]:

# Statistical Summary
df.describe(include="all")

,review_id,product_name,category,rating,review_text,true_sentiment,reviewer_age,helpful_votes
count,1230.000000,1230,1230,1220.000000,1210,1230,1230.000000,1230.000000
unique,NaN,12,6,NaN,245,3,NaN,NaN
top,NaN,Face Cream,Clothing,NaN,Best purchase I have made this year. Highly re...,Positive,NaN,NaN
freq,NaN,120,235,NaN,35,617,NaN,NaN
mean,599.997561,NaN,NaN,3.320492,NaN,NaN,41.112195,23.956098
std,346.052667,NaN,NaN,1.375303,NaN,NaN,13.383087,14.290079
min,1.000000,NaN,NaN,1.000000,NaN,NaN,18.000000,0.000000
25%,300.250000,NaN,NaN,2.000000,NaN,NaN,30.000000,12.000000
50%,598.500000,NaN,NaN,4.000000,NaN,NaN,42.000000,23.000000
75%,899.750000,NaN,NaN,5.000000,NaN,NaN,53.000000,36.000000


In [7]:
# Missing Values
df.isnull().sum()


review_id          0
product_name       0
category           0
rating            10
review_text       20
true_sentiment     0
reviewer_age       0
helpful_votes      0
dtype: int64

In [8]:
# Duplicate Rows
df.duplicated().sum()


np.int64(29)

- review_text is our main feature → remove rows with missing reviews.
- rating is mainly for EDA → fill missing ratings with the median.

In [9]:
# Remove rows where review text is missing
df.dropna(subset=["review_text"], inplace=True)

# Fill missing ratings with median
df["rating"] = df["rating"].fillna(df["rating"].median())

# Verify
df.isnull().sum()

review_id         0
product_name      0
category          0
rating            0
review_text       0
true_sentiment    0
reviewer_age      0
helpful_votes     0
dtype: int64

In [10]:
print("Shape before removing duplicate reviews:", df.shape)

df.drop_duplicates(subset=["review_text"], inplace=True)

df.reset_index(drop=True, inplace=True)

print("Shape after removing duplicate reviews:", df.shape)

Shape before removing duplicate reviews: (1210, 8)
Shape after removing duplicate reviews: (245, 8)


In [11]:
df["review_text"].duplicated().sum()

np.int64(0)

In [12]:
df= df.drop(columns=[
    "review_id",
    "reviewer_age",
    "helpful_votes"
])

df.head()

,product_name,category,rating,review_text,true_sentiment
0,Laptop,Books,5.0,Super happy with this purchase. Packaging was ...,Positive
1,Laptop,Home & Kitchen,5.0,Works like a charm. Very happy with the overal...,Positive
2,Jeans,Beauty,5.0,Fantastic product at a very reasonable price p...,Positive
3,Blender,Sports,5.0,Works like a charm. Very happy with the overal...,Positive
4,Yoga Mat,Books,4.0,Works like a charm. Very happy with the overal...,Positive


In [13]:
#Text Preprocessing
#Cell 1 – Import NLP Libraries
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [14]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\91866\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\91866\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\91866\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [15]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

In [16]:
def clean_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove HTML tags
    text = re.sub(r"<.*?>", "", text)

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenize
    words = text.split()

    # Remove stopwords and lemmatize
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)

In [17]:
# Apply Cleaning
df["clean_review"] = df["review_text"].apply(clean_text)

df.head()

,product_name,category,rating,review_text,true_sentiment,clean_review
0,Laptop,Books,5.0,Super happy with this purchase. Packaging was ...,Positive,super happy purchase packaging also excellent ...
1,Laptop,Home & Kitchen,5.0,Works like a charm. Very happy with the overal...,Positive,work like charm happy overall experience packa...
2,Jeans,Beauty,5.0,Fantastic product at a very reasonable price p...,Positive,fantastic product reasonable price point
3,Blender,Sports,5.0,Works like a charm. Very happy with the overal...,Positive,work like charm happy overall experience
4,Yoga Mat,Books,4.0,Works like a charm. Very happy with the overal...,Positive,work like charm happy overall experience deliv...


In [18]:
df.duplicated().sum()

np.int64(0)

In [19]:
# Compare Before & After
df[['review_text','clean_review']].head(10)

,review_text,clean_review
0,Super happy with this purchase. Packaging was ...,super happy purchase packaging also excellent ...
1,Works like a charm. Very happy with the overal...,work like charm happy overall experience packa...
2,Fantastic product at a very reasonable price p...,fantastic product reasonable price point
3,Works like a charm. Very happy with the overal...,work like charm happy overall experience
4,Works like a charm. Very happy with the overal...,work like charm happy overall experience deliv...
5,Best purchase I have made this year. Highly re...,best purchase made year highly recommend impre...
6,The product is durable and looks exactly like ...,product durable look exactly like picture
7,Best purchase I have made this year. Highly re...,best purchase made year highly recommend recom...
8,Excellent build quality and quick shipping. Fi...,excellent build quality quick shipping five star
9,Great value for money. The product works perfe...,great value money product work perfectly


In [20]:
# Remove Empty Reviews
df = df[df["clean_review"] != ""]

df.reset_index(drop=True, inplace=True)

print(df.shape)

(245, 6)


In [21]:
df.to_csv("cleaned_reviews.csv", index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


### Feature Engineering

In [22]:
# Select Features and Target
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    max_features=5000,
    ngram_range=(1,2)
)

X = tfidf.fit_transform(df["review_text"])

y = df["true_sentiment"]

In [23]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y = le.fit_transform(df["true_sentiment"])

In [24]:
print(dict(zip(le.classes_, le.transform(le.classes_))))

{'Negative': np.int64(0), 'Neutral': np.int64(1), 'Positive': np.int64(2)}


In [25]:
# Train-Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [26]:
#Verify the Split
print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)

print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)



X_train : (196, 459)
X_test  : (49, 459)
y_train : (196,)
y_test  : (49,)


### Model 1 – Logistic Regression

In [27]:
sample_reviews = [

    "The phone is amazing. Battery backup is excellent.",

    "The phone is okay. Performance is average.",

    "Worst mobile ever. Waste of money.",

    "Camera quality is good but battery drains quickly.",

    "Packaging was damaged but the product works fine.",

    "Excellent value for money. Highly recommended."
]

In [29]:
from sklearn.linear_model import LogisticRegression

In [30]:
# Train the Model
lr = LogisticRegression(random_state=42)

lr.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [31]:
# Prediction
y_pred_lr = lr.predict(X_test)

In [32]:
# Accuracy Score
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred_lr)

print("Accuracy :", round(accuracy,4))

Accuracy : 1.0


In [33]:
sample_vector = tfidf.transform(sample_reviews)

predictions = lr.predict(sample_vector)

predicted_sentiments = le.inverse_transform(predictions)

result_df = pd.DataFrame({

    "Review": sample_reviews,

    "Predicted Sentiment": predicted_sentiments

})

result_df

,Review,Predicted Sentiment
0,The phone is amazing. Battery backup is excell...,Positive
1,The phone is okay. Performance is average.,Neutral
2,Worst mobile ever. Waste of money.,Negative
3,Camera quality is good but battery drains quic...,Positive
4,Packaging was damaged but the product works fine.,Positive
5,Excellent value for money. Highly recommended.,Positive


In [29]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_lr))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        16
           1       1.00      1.00      1.00        15
           2       1.00      1.00      1.00        18

    accuracy                           1.00        49
   macro avg       1.00      1.00      1.00        49
weighted avg       1.00      1.00      1.00        49



In [30]:
df.groupby("review_text")["true_sentiment"].nunique().value_counts()

true_sentiment
1    245
Name: count, dtype: int64

In [31]:
# Multinomial Naive Bayes
#Train the Model
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()

nb.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [32]:
# Predict
y_pred_nb = nb.predict(X_test)


In [33]:
#Accuracy
from sklearn.metrics import accuracy_score

accuracy_nb = accuracy_score(y_test, y_pred_nb)

print("Accuracy :", round(accuracy_nb,4))

Accuracy : 1.0


In [34]:
# Linear SVM
#Import SVM
from sklearn.svm import LinearSVC

In [35]:
# Train Model
svm = LinearSVC(random_state=42)

svm.fit(X_train, y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,verbose,0
,random_state,42


In [36]:
#Prediction
y_pred_svm = svm.predict(X_test)

In [37]:
# Accuracy
from sklearn.metrics import accuracy_score

accuracy_svm = accuracy_score(y_test, y_pred_svm)

print("Accuracy :", round(accuracy_svm,4))

Accuracy : 1.0


In [38]:
# Classification Report
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_svm))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        16
           1       1.00      1.00      1.00        15
           2       1.00      1.00      1.00        18

    accuracy                           1.00        49
   macro avg       1.00      1.00      1.00        49
weighted avg       1.00      1.00      1.00        49



In [39]:
# XGBoost

from xgboost import XGBClassifier

xgb = XGBClassifier(
    objective="multi:softmax",
    num_class=3,
    random_state=42,
    eval_metric="mlogloss"
)

xgb.fit(X_train, y_train)

,objective,'multi:softmax'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'mlogloss'


In [40]:
# prediction
y_pred_xgb = xgb.predict(X_test)

In [41]:
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)

print("Accuracy :", round(accuracy_xgb,4))

Accuracy : 0.9592


In [42]:
results = pd.DataFrame({

    "Model":[
        "Logistic Regression",
        "Multinomial Naive Bayes",
        "Linear SVM",
        "XGBoost"
    ],

    "Accuracy":[
        accuracy,
        accuracy_nb,
        accuracy_svm,
        accuracy_xgb
    ]
})

results

,Model,Accuracy
0,Logistic Regression,1.000000
1,Multinomial Naive Bayes,1.000000
2,Linear SVM,1.000000
3,XGBoost,0.959184


In [43]:
# random forest
from sklearn.ensemble import RandomForestClassifier

In [44]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

In [45]:
rf.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [46]:
y_pred_rf = rf.predict(X_test)

In [47]:
from sklearn.metrics import accuracy_score

accuracy_rf = accuracy_score(y_test, y_pred_rf)

print("Accuracy :", round(accuracy_rf, 4))

Accuracy : 1.0


In [62]:
import pickle

# Load the saved model
with open("sentiment_model.pkl", "rb") as file:
    model = pickle.load(file)

# Load the TF-IDF vectorizer
with open("tfidf_vectorizer.pkl", "rb") as file:
    tfidf = pickle.load(file)

# Load the Label Encoder
with open("label_encoder.pkl", "rb") as file:
    le = pickle.load(file)


In [68]:
print("X_test shape :", X_test.shape)
print("y_test shape :", y_test.shape)
print("y_pred_rf shape :", y_pred_rf.shape)

X_test shape : (2, 459)
y_test shape : (235,)
y_pred_rf shape : (2,)
